#### Task 2: Total sales (Quantity * UnitPrice) by Country - the pandas equivalent of the Excel PivotTable.

In [2]:
import os
import numpy as np
import pandas as pd
 
HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
 
 
def load_data():
    """Load the real OnlineRetail.csv if it is in this folder, otherwise the simulated sample."""
    for name in ("OnlineRetail.csv", "OnlineRetail_sample.csv"):
        path = os.path.join(HERE, name)
        if os.path.exists(path):
            df = pd.read_csv(path, encoding="ISO-8859-1", dtype={"InvoiceNo": str, "StockCode": str})
            print(f"Loaded {name}: {len(df):,} rows" + ("" if name == "OnlineRetail.csv" else "  (SIMULATED STAND-IN DATA, not the real file)"))
            break
    else:
        raise FileNotFoundError("Put OnlineRetail.csv (or run generate_sample_data.py) in the same folder.")
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    df["LineTotal"] = df["Quantity"] * df["UnitPrice"]
    return df
 
 
def valid_rows(df):
    """Drop cancellations (InvoiceNo starts with C) and rows with non-positive quantity or price."""
    return df[~df["InvoiceNo"].str.startswith("C") & (df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
 
 
df = valid_rows(load_data())              # valid sales only (no cancellations / returns)
 
# Excel equivalent: PivotTable with Country in Rows and Sum of LineTotal in Values
pivot = (df.pivot_table(index="Country", values="LineTotal", aggfunc="sum")
           .sort_values("LineTotal", ascending=False))
pivot["Share %"] = pivot["LineTotal"] / pivot["LineTotal"].sum() * 100
print("\nTotal sales by country (sorted):")
print(pivot.round(2).to_string())
 
top3 = pivot.head(3).index.tolist()
print(f"\nTop 3 countries by total sales: 1) {top3[0]}  2) {top3[1]}  3) {top3[2]}")
others = pivot.drop(index="United Kingdom", errors="ignore").head(3).index.tolist()
print(f"Top 3 excluding the United Kingdom: {', '.join(others)}")

Loaded OnlineRetail_sample.csv: 3,989 rows  (SIMULATED STAND-IN DATA, not the real file)

Total sales by country (sorted):
                LineTotal  Share %
Country                           
United Kingdom  207505.63    66.11
Germany          47103.16    15.01
EIRE             18306.74     5.83
Australia        15825.90     5.04
Netherlands       8421.42     2.68
France            5988.97     1.91
Switzerland       5773.04     1.84
Belgium           4804.00     1.53
Spain              171.26     0.05

Top 3 countries by total sales: 1) United Kingdom  2) Germany  3) EIRE
Top 3 excluding the United Kingdom: Germany, EIRE, Australia
